# 1. Pool Overview — QA

**Purpose** — Cross-mart smoke test for the indexed pool universe. Validates that every downstream notebook can rely on the same pool reference.

**Source tables**
- `dim_indexed_pools`
- `stg_pools`
- `mart_tvl_current`
- `int_current_state`
- `int_lp_macro_daily`
- `mart_tvl_daily`
- `mart_volume_daily`
- `mart_price_analytics_daily`

**Window** — data span ends `2026-04-06`. The bootstrap cell below sets `END_DATE` and two convenience starts (`START_DATE_30D`, `START_DATE_90D`).

**Convention** — values denominated in `token0` / `token0_equiv`. For the indexed USDC/WETH 0.05% pool, `token0 = USDC` so `token0_equiv ≈ USD`.

This notebook is a **QA workbench**, not a PM dashboard. Each section ends with explicit PASS/WARN/FAIL checks. Final §4 cell prints the verdict.


In [1]:
import sys, os
from pathlib import Path

ROOT = Path(os.getcwd()).resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

os.environ.setdefault("CLICKHOUSE_HOST", "localhost")
os.environ.setdefault("CLICKHOUSE_PORT", "8124")
os.environ.setdefault("CLICKHOUSE_DATABASE", "analytics")
os.environ.setdefault("CLICKHOUSE_USER", "default")
os.environ.setdefault("CLICKHOUSE_PASSWORD", "1234")

import polars as pl
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import date, timedelta

from core.db import ClickHouseDB, ClickHouseSettings
db = ClickHouseDB.from_settings(ClickHouseSettings())

POOL      = "0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640"
END_DATE  = "2026-04-06"            # Last date with full mart coverage
START_DATE_90D = (date.fromisoformat(END_DATE) - timedelta(days=90)).isoformat()
START_DATE_30D = (date.fromisoformat(END_DATE) - timedelta(days=30)).isoformat()

print(f"Pool:       {POOL}")
print(f"End date:   {END_DATE}")
print(f"Window 90d: {START_DATE_90D} → {END_DATE}")
print(f"Window 30d: {START_DATE_30D} → {END_DATE}")

# Auto-cleanup: close CH connection on kernel shutdown so sockets don't pile up
import atexit as _atexit
_atexit.register(lambda: db.close() if 'db' in globals() and db is not None else None)


Pool:       0x88e6a0c2ddd26feeb64f039a2c41296fcb3f5640
End date:   2026-04-06
Window 90d: 2026-01-06 → 2026-04-06
Window 30d: 2026-03-07 → 2026-04-06


<function __main__.<lambda>()>

In [2]:
# QA check helper — records pass/warn/fail into QA_RESULTS for the §4 verdict cell.
QA_RESULTS = []

def qa_check(label, passed, severity="fail", detail=""):
    if passed:
        status, icon = "PASS", "🟢"
    elif severity == "warn":
        status, icon = "WARN", "🟠"
    else:
        status, icon = "FAIL", "🔴"
    QA_RESULTS.append({"label": label, "status": status, "severity": severity, "detail": str(detail)})
    print(f"{icon} {status:4s} | {label}" + (f"  —  {detail}" if detail else ""))

def qa_table_exists(name, database="analytics"):
    try:
        n = db.read_query(
            f"SELECT count() AS n FROM system.tables WHERE database='{database}' AND name='{name}'"
        )["n"][0]
        return int(n) > 0
    except Exception:
        return False

def qa_safe(query):
    try:
        return db.read_query(query)
    except Exception as e:
        return None


## §1 — Data fetch & coverage

In [3]:
# Load static + daily marts. Schema dump + coverage matrix.
identity   = db.read_query("SELECT * FROM dim_indexed_pools WHERE lower(pool_address) = lower(%(p)s)" % {"p": repr(POOL)})
pool_meta  = db.read_query(f"SELECT * FROM stg_pools WHERE lower(pool) = lower('{POOL}')")
current    = db.read_query(f"SELECT * FROM mart_tvl_current WHERE lower(pool) = lower('{POOL}')")
state      = db.read_query(f"SELECT * FROM int_current_state WHERE lower(pool) = lower('{POOL}')")
macro      = db.read_query(f"SELECT * FROM int_lp_macro_daily WHERE lower(pool) = lower('{POOL}') AND date >= toDate('{START_DATE_90D}') AND date <= toDate('{END_DATE}') ORDER BY date")
print(f"identity rows={identity.height}  meta rows={pool_meta.height}  current rows={current.height}  state rows={state.height}  macro rows={macro.height}")
identity

identity rows=1  meta rows=1  current rows=1  state rows=1  macro rows=91


pool_address,chain,dex,token0_address,token1_address,token0_symbol,token1_symbol,token0_decimals,token1_decimals,fee_tier,tick_spacing,is_active,indexed_since_block,indexed_until_block,coverage_status,created_at,updated_at
str,str,str,str,str,str,str,i64,i64,f64,i64,i64,i64,null,str,datetime[μs],datetime[μs]
"""0x88e6A0c2dDD26FEEb64F039a2c41…","""ethereum""","""uniswap_v3""","""0xA0b86991c6218b36c1d19D4a2e9E…","""0xC02aaA39b223FE8D0A0e5C4F27eA…","""USDC""","""WETH""",6,18,0.05,10,1,12376729,null,"""active""",2021-05-05 21:42:11,2026-04-28 00:45:56


In [4]:
# Coverage matrix: pool present per daily mart over the loaded window.
COVERAGE_MARTS = ["mart_tvl_daily", "mart_volume_daily", "mart_price_analytics_daily", "int_lp_macro_daily"]
cov_rows = []
for m in COVERAGE_MARTS:
    df = db.read_query(f"SELECT min(date) AS min_date, max(date) AS max_date, count() AS rows, countDistinct(date) AS distinct_dates FROM {m} WHERE lower(pool) = lower('{POOL}')")
    cov_rows.append({"mart": m, **df.row(0, named=True)})
coverage = pl.DataFrame(cov_rows)
coverage

mart,min_date,max_date,rows,distinct_dates
str,date,date,i64,i64
"""mart_tvl_daily""",2021-05-05,2026-04-06,1798,1798
"""mart_volume_daily""",2021-05-05,2026-04-06,1798,1798
"""mart_price_analytics_daily""",2021-05-05,2026-04-16,1808,1808
"""int_lp_macro_daily""",2021-05-05,2026-04-06,1798,1798


## §2 — Sanity checks

In [5]:
# Pool universe presence
qa_check("pool present in dim_indexed_pools", identity.height == 1, detail=f"rows={identity.height}")
qa_check("pool present in stg_pools",         pool_meta.height == 1, detail=f"rows={pool_meta.height}")
qa_check("pool present in int_current_state", state.height == 1,     detail=f"rows={state.height}")
qa_check("pool present in mart_tvl_current",  current.height == 1,   detail=f"rows={current.height}")
for m in COVERAGE_MARTS:
    n = db.read_query(f"SELECT count() AS n FROM {m} WHERE lower(pool) = lower('{POOL}')")["n"][0]
    qa_check(f"pool present in {m}", int(n) > 0, detail=f"rows={int(n)}")

🟢 PASS | pool present in dim_indexed_pools  —  rows=1
🟢 PASS | pool present in stg_pools  —  rows=1
🟢 PASS | pool present in int_current_state  —  rows=1
🟢 PASS | pool present in mart_tvl_current  —  rows=1
🟢 PASS | pool present in mart_tvl_daily  —  rows=1798
🟢 PASS | pool present in mart_volume_daily  —  rows=1798
🟢 PASS | pool present in mart_price_analytics_daily  —  rows=1808
🟢 PASS | pool present in int_lp_macro_daily  —  rows=1798


In [6]:
# Date coverage and uniqueness on daily marts
for m in COVERAGE_MARTS:
    df = db.read_query(f"SELECT count() AS rows, countDistinct(date) AS distinct_dates, max(date) AS max_d FROM {m} WHERE lower(pool) = lower('{POOL}')")
    rows, distinct_d, max_d = int(df['rows'][0]), int(df['distinct_dates'][0]), df['max_d'][0]
    qa_check(f"{m}: no duplicate (pool, date)", rows == distinct_d, detail=f"rows={rows} distinct_dates={distinct_d}")
    qa_check(f"{m}: max date >= {END_DATE} - 1d",
             max_d >= date.fromisoformat(END_DATE) - timedelta(days=1),
             severity="warn",
             detail=f"max_date={max_d}")

🟢 PASS | mart_tvl_daily: no duplicate (pool, date)  —  rows=1798 distinct_dates=1798
🟢 PASS | mart_tvl_daily: max date >= 2026-04-06 - 1d  —  max_date=2026-04-06
🟢 PASS | mart_volume_daily: no duplicate (pool, date)  —  rows=1798 distinct_dates=1798
🟢 PASS | mart_volume_daily: max date >= 2026-04-06 - 1d  —  max_date=2026-04-06
🟢 PASS | mart_price_analytics_daily: no duplicate (pool, date)  —  rows=1808 distinct_dates=1808
🟢 PASS | mart_price_analytics_daily: max date >= 2026-04-06 - 1d  —  max_date=2026-04-16
🟢 PASS | int_lp_macro_daily: no duplicate (pool, date)  —  rows=1798 distinct_dates=1798
🟢 PASS | int_lp_macro_daily: max date >= 2026-04-06 - 1d  —  max_date=2026-04-06


In [7]:
# Static / state sanity
qa_check("current_tick non-null", state['current_tick'][0] is not None,
         detail=f"current_tick={state['current_tick'][0]}")
qa_check("price_token1_in_token0 > 0",
         float(state['price_token1_in_token0'][0]) > 0,
         detail=f"price={state['price_token1_in_token0'][0]}")
qa_check("fee_tier stable in stg_pools (single value)",
         pool_meta.height == 1,
         detail=f"fee_bps={pool_meta['fee_bps'][0]}")

🟢 PASS | current_tick non-null  —  current_tick=199587
🟢 PASS | price_token1_in_token0 > 0  —  price=2150.092004
🟢 PASS | fee_tier stable in stg_pools (single value)  —  fee_bps=0.05000000074505806


In [8]:
# Non-negativity + jump detection on daily marts
tvl_neg = db.read_query(f"SELECT countIf(tvl_token0_equiv < 0) AS n FROM mart_tvl_daily WHERE lower(pool) = lower('{POOL}')")["n"][0]
vol_neg = db.read_query(f"SELECT countIf(volume_token0_equiv < 0) AS n FROM mart_volume_daily WHERE lower(pool) = lower('{POOL}')")["n"][0]
qa_check("mart_tvl_daily.tvl_token0_equiv >= 0",  int(tvl_neg) == 0, detail=f"violations={int(tvl_neg)}")
qa_check("mart_volume_daily.volume_token0_equiv >= 0", int(vol_neg) == 0, detail=f"violations={int(vol_neg)}")

jump_q = f"""
SELECT count() AS n FROM (
  SELECT date,
         tvl_token0_equiv,
         lagInFrame(tvl_token0_equiv, 1, NULL) OVER (PARTITION BY pool ORDER BY date) AS prev_tvl
  FROM mart_tvl_daily
  WHERE lower(pool) = lower('{POOL}') AND date >= toDate('{START_DATE_90D}')
) WHERE prev_tvl > 0 AND abs(tvl_token0_equiv - prev_tvl) / prev_tvl > 0.5
"""
n_jumps = db.read_query(jump_q)["n"][0]
qa_check("no daily TVL jumps > 50% over last 90d",
         int(n_jumps) == 0, severity="warn",
         detail=f"jump_days={int(n_jumps)}")

🟢 PASS | mart_tvl_daily.tvl_token0_equiv >= 0  —  violations=0
🟢 PASS | mart_volume_daily.volume_token0_equiv >= 0  —  violations=0
🟢 PASS | no daily TVL jumps > 50% over last 90d  —  jump_days=0


## §3 — Diagnostic visuals

**QA purpose**: confirm the pool exists in every expected daily mart and that date coverage is contiguous up to `END_DATE`.

In [9]:
fig = px.imshow(
    [[1] * len(COVERAGE_MARTS)],
    x=COVERAGE_MARTS, y=["coverage"], color_continuous_scale=[[0, "lightgray"], [1, "#22c55e"]],
    aspect="auto", title="Pool coverage matrix (1 = present)",
)
fig.update_layout(height=160, coloraxis_showscale=False)
fig.show()
coverage

mart,min_date,max_date,rows,distinct_dates
str,date,date,i64,i64
"""mart_tvl_daily""",2021-05-05,2026-04-06,1798,1798
"""mart_volume_daily""",2021-05-05,2026-04-06,1798,1798
"""mart_price_analytics_daily""",2021-05-05,2026-04-16,1808,1808
"""int_lp_macro_daily""",2021-05-05,2026-04-06,1798,1798


**QA purpose**: smoke-test that TVL / volume / fees / vol load cohesively on the same daily axis. Outliers visible at a glance.

In [10]:
fig = make_subplots(rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.04,
    subplot_titles=("TVL (token0_equiv)", "Volume (token0_equiv)", "Fees (token0_equiv)", "Realized vol 7d"))
ds = macro["date"].to_list()
fig.add_trace(go.Scatter(x=ds, y=macro["tvl_token0_equiv"].to_list(), name="TVL"), row=1, col=1)
fig.add_trace(go.Bar    (x=ds, y=macro["volume_token0_equiv"].to_list(), name="Volume"), row=2, col=1)
fig.add_trace(go.Bar    (x=ds, y=macro["fees_token0_equiv"].to_list(),  name="Fees"),   row=3, col=1)
fig.add_trace(go.Scatter(x=ds, y=macro["realized_vol_7d"].to_list(),    name="σ 7d"),   row=4, col=1)
fig.update_layout(height=600, showlegend=False, title_text="Pool overview — quad smoke chart")
fig.show()

**QA purpose**: detect TVL discontinuities (jumps > 50% day-over-day) — annotate dates, then cross-check vs LP flow in 03.

In [11]:
jumps = db.read_query(f"""
SELECT date, tvl_token0_equiv,
       lagInFrame(tvl_token0_equiv, 1, NULL) OVER (PARTITION BY pool ORDER BY date) AS prev_tvl,
       (tvl_token0_equiv - lagInFrame(tvl_token0_equiv, 1, NULL) OVER (PARTITION BY pool ORDER BY date)) / nullIf(lagInFrame(tvl_token0_equiv, 1, NULL) OVER (PARTITION BY pool ORDER BY date), 0) AS pct_change
FROM mart_tvl_daily
WHERE lower(pool) = lower('{POOL}') AND date >= toDate('{START_DATE_90D}')
ORDER BY date
""")
fig = go.Figure()
fig.add_trace(go.Scatter(x=jumps["date"].to_list(), y=jumps["tvl_token0_equiv"].to_list(), mode="lines", name="TVL"))
flagged = jumps.filter(pl.col("pct_change").abs() > 0.5)
if flagged.height > 0:
    fig.add_trace(go.Scatter(x=flagged["date"].to_list(), y=flagged["tvl_token0_equiv"].to_list(),
                              mode="markers", marker=dict(size=10, color="red"), name=">50% jump"))
fig.update_layout(height=320, title_text="TVL daily — jump audit")
fig.show()
print(f"Flagged jump days: {flagged.height}")

Flagged jump days: 0


**QA purpose**: period summary KPIs — quick eyeballable totals to compare with notebooks 04 (fees), 03 (TVL/flow).

In [12]:
last_day = macro.tail(1).to_dicts()[0]
print(f"Window: {macro['date'].min()} → {macro['date'].max()}  ({macro.height} days)")
print(f"Latest TVL:                  {last_day['tvl_token0_equiv']:>16,.2f}")
print(f"Latest active TVL:           {last_day['active_tvl_token0']:>16,.2f}")
print(f"Latest capital_utilization:  {last_day['capital_utilization']:>16.2%}")
print(f"90d cum volume:              {macro['volume_token0_equiv'].sum():>16,.2f}")
print(f"90d cum fees:                {macro['fees_token0_equiv'].sum():>16,.2f}")
print(f"90d swap count:              {int(macro['swap_count'].sum()):>16,}")

Window: 2026-01-06 → 2026-04-06  (91 days)
Latest TVL:                     93,623,508.29
Latest active TVL:              74,604,437.77
Latest capital_utilization:            79.69%
90d cum volume:              16,977,156,575.27
90d cum fees:                    8,488,578.41
90d swap count:                       584,535


## §4 — QA verdict

In [13]:
# ─── §4 QA verdict ──────────────────────────────────────────────────────────
verdict = pl.DataFrame(QA_RESULTS) if QA_RESULTS else pl.DataFrame({"label": [], "status": [], "severity": [], "detail": []})
n_pass = verdict.filter(pl.col("status") == "PASS").height
n_warn = verdict.filter(pl.col("status") == "WARN").height
n_fail = verdict.filter(pl.col("status") == "FAIL").height

if n_fail > 0:
    overall = "FAIL"
elif n_warn > 0:
    overall = "WARN"
else:
    overall = "PASS"

print("=" * 70)
print(f"  QA VERDICT — overall: {overall}")
print(f"  PASS: {n_pass}   WARN: {n_warn}   FAIL: {n_fail}")
print("=" * 70)
if n_fail > 0:
    print("\nFailures:")
    for row in verdict.filter(pl.col("status") == "FAIL").to_dicts():
        print(f"  🔴 {row['label']}  —  {row['detail']}")
if n_warn > 0:
    print("\nWarnings:")
    for row in verdict.filter(pl.col("status") == "WARN").to_dicts():
        print(f"  🟠 {row['label']}  —  {row['detail']}")

verdict


  QA VERDICT — overall: PASS
  PASS: 22   WARN: 0   FAIL: 0


label,status,severity,detail
str,str,str,str
"""pool present in dim_indexed_po…","""PASS""","""fail""","""rows=1"""
"""pool present in stg_pools""","""PASS""","""fail""","""rows=1"""
"""pool present in int_current_st…","""PASS""","""fail""","""rows=1"""
"""pool present in mart_tvl_curre…","""PASS""","""fail""","""rows=1"""
"""pool present in mart_tvl_daily""","""PASS""","""fail""","""rows=1798"""
…,…,…,…
"""price_token1_in_token0 > 0""","""PASS""","""fail""","""price=2150.092004"""
"""fee_tier stable in stg_pools (…","""PASS""","""fail""","""fee_bps=0.05000000074505806"""
"""mart_tvl_daily.tvl_token0_equi…","""PASS""","""fail""","""violations=0"""
